# Track 1 · 05 Train — ClipLoss probe, then LoRA

Two rungs in one file, because the second warm-starts from the first.

**Run `track1_02_features` first, in this session.** Section A needs
`features_pooled.npz` from Drive; Section B needs the EEG, which it re-epochs itself.

---

## Why this exists

The ridge head scored exactly chance (rank 0.4998, top-5 2.48% against a 2.50% floor),
and the scaling curve was flat from 3 to 12 subjects, so the problem is not data. Two
deviations from the reference remained:

**Wrong objective.** Ridge minimises `||pred - target||^2`. The metric is cosine
retrieval. Because DINOv2 embeddings share a large common component, the cheapest way to
reduce squared error is to predict something near the gallery mean, which is exactly what
we watched happen: a collapsed head scored 0.78 cosine while being useless. The reference
uses `ClipLoss` with `norm_kind: y`, `temperature: false`, `symmetric: false` — contrastive,
so it optimises *distinguishability* rather than absolute closeness.

**Wrong rung.** The reference recipe is "2-step (linear probe then LoRA on QKVO)". The
leaderboard agrees: an entry labelled `test reve 3 epochs` scores 0.1349, five times
chance, from three epochs of fine-tuning. Our frozen probe sits at chance.

Section A fixes the objective while everything stays frozen. Section B unfreezes the
attention projections. A is minutes on cached features; B is the expensive one.

## What actually trains

| | Section A | Section B |
|---|---|---|
| trainable | head, ~0.79M | head + LoRA, ~1.9M |
| frozen | all 69M encoder | all 69M encoder (LoRA adds, never overwrites) |
| input | cached 512-d features | raw windows, re-epoched |
| feature cache | used | useless, encoder changes every step |

LoRA freezes each `W` and learns `B·A` beside it with `B` initialised to zero, so training
starts exactly at the pretrained function and departs from there. Targets are
`transformer.layers.*.0.{to_qkv,to_out}`, the 44 attention projections named in
`reve.yaml`.

## One detail that would silently corrupt the loss

About 30% of windows repeat an image. Two windows of the *same* image landing in one batch
become negatives of each other, so the loss punishes the model for being right. Those pairs
are masked out.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

RUN_CLIP_PROBE = True
RUN_LORA = True
MAKE_ZIP = True

SEED = 33
VAL_IMAGE_FRAC = 0.2

# Section A
A_EPOCHS, A_BATCH, A_LR = 50, 256, 1e-3

# Section B
B_EPOCHS, B_BATCH, B_LR, B_HEAD_LR = 8, 32, 1e-4, 1e-3
LORA_RANK, LORA_ALPHA = 16, 32

NORM_PRED = False      # reve.yaml norm_kind: y -> normalise the target only
ENCODER = 'brain-bzh/reve-base'
SUBMISSION_NAME = 'track1_reve_lora'
ZIP_DIR = '/content/submission'

In [ ]:
%pip install -q mne 'braindecode>=1.8' huggingface_hub safetensors

import importlib
for mod in ('mne', 'braindecode', 'torch'):
    m = importlib.import_module(mod)
    print(f'{mod:12} {getattr(m, "__version__", "ok")}')

In [ ]:
import os, json, time, shutil, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
PREP, RAW, WEIGHTS = BASE / 'prepared', BASE / 'raw', BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
DATASET = 'nm000134'


def report(title, checks):
    print(title)
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  -> {sum(c[1] for c in checks)} ok, {sum(not c[1] for c in checks)} failed')


def l2np(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def rank_pct(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2np(pred) @ l2np(emb[gal]).T
    tc = sims[np.arange(len(sims)), truth]
    return float((sims < tc[:, None]).mean())


def top5(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2np(pred) @ l2np(emb[gal]).T
    k = min(5, len(gal))
    topk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, topk)]), 100.0 * k / len(gal)


def score(pred, y_ids, subj, emb, label):
    """Per-window and within-subject-aggregated. Warm-up scores per-window."""
    acc, ch = top5(pred, y_ids, emb)
    r = rank_pct(pred, y_ids, emb)
    agg = {}
    for p_, y_, s_ in zip(pred, y_ids, subj):
        agg.setdefault((s_, y_), []).append(p_)
    ks = sorted(agg)
    pa = np.stack([np.mean(agg[k], 0) for k in ks])
    ya = np.array([k[1] for k in ks])
    acc_a, _ = top5(pa, ya, emb)
    print(f'  {label:<16} rank {r:.4f}   top-5 {acc:5.2f}% (per-window, = the warm-up '
          f'metric)   {acc_a:5.2f}% (subject-agg)   chance {ch:.2f}%')
    return dict(rank=r, top5=acc, top5_agg=acc_a, chance=ch)


print(f'device {DEVICE}')
if DEVICE == 'cpu':
    print('No GPU. Section A is fine on CPU; Section B will be painfully slow.')

## Load the cached features and the target table

In [ ]:
f = np.load(PREP / 'features_pooled.npz', allow_pickle=True)
t = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
emb = l2np(t['emb'].astype('float32'))          # ClipLoss norm_kind: y
DIM = emb.shape[1]

Xtr, ytr, str_ = f['X_train'].astype('float32'), f['y_train'], f['subj_train']
Xte, yte, ste = f['X_test'].astype('float32'), f['y_test'], f['subj_test']

# Preprocessing settings travel with the artefact, so Section B cannot drift from
# whatever track1_02_features actually did.
PP = dict(filt=[float(x) for x in np.atleast_1d(f['filt'])] if f['filt'].size else None,
          baseline=str(f['baseline']), clamp=float(f['clamp']),
          scale_scope=str(f['scale_scope']), sfreq=float(f['sfreq']))
print('features', Xtr.shape, Xte.shape, ' targets', emb.shape)
print('preprocessing recorded in the bundle:', PP)
if PP['filt'] is None:
    print('\nWARNING: these features were built with no band-pass. Re-run')
    print('track1_02_features before trusting anything below.')

# image-wise validation split, so lambda/early stopping never sees a training image
imgs = np.unique(ytr)
val_imgs = set(rng.choice(imgs, int(len(imgs) * VAL_IMAGE_FRAC), replace=False).tolist())
is_val = np.array([y in val_imgs for y in ytr])
print(f'{len(imgs):,} train images -> {len(val_imgs):,} held out '
      f'({(~is_val).sum():,} fit / {is_val.sum():,} val windows)')

## Section A — ClipLoss on the frozen head

Same inputs and same `512 -> 1536` linear as the ridge fit. Only the objective changes.

In [ ]:
EMB_T = torch.tensor(emb, device=DEVICE)


def clip_loss(pred, target_idx):
    """Contrastive, brain-to-target only, fixed temperature, target-normalised.

    Mirrors reve.yaml: norm_kind y, temperature false, symmetric false.
    """
    Y = EMB_T[target_idx]                       # already unit norm
    P = F.normalize(pred, dim=1) if NORM_PRED else pred
    logits = P @ Y.T
    same = target_idx[:, None] == target_idx[None, :]
    same.fill_diagonal_(False)                  # repeats of one image are NOT negatives
    logits = logits.masked_fill(same, -1e4)
    return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


def train_head(Xf, yf, Xv, yv, epochs, batch, lr, head=None, log_every=10):
    Xf_t = torch.tensor(Xf, device=DEVICE)
    yf_t = torch.tensor(yf.astype('int64'), device=DEVICE)
    Xv_t = torch.tensor(Xv, device=DEVICE)
    head = head or nn.Linear(Xf.shape[1], DIM).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-2)
    best, best_state = -1, None
    for ep in range(1, epochs + 1):
        head.train()
        perm = torch.randperm(len(Xf_t), device=DEVICE)
        tot = 0.0
        for i in range(0, len(perm) - 1, batch):
            idx = perm[i:i + batch]
            if len(idx) < 8:
                continue
            opt.zero_grad()
            loss = clip_loss(head(Xf_t[idx]), yf_t[idx])
            loss.backward(); opt.step()
            tot += float(loss) * len(idx)
        head.eval()
        with torch.no_grad():
            pv = head(Xv_t).cpu().numpy()
        r = rank_pct(pv, yv, emb)
        if r > best:
            best, best_state = r, {k: v.detach().clone() for k, v in head.state_dict().items()}
        if ep % log_every == 0 or ep == 1:
            print(f'    epoch {ep:3d}  loss {tot / len(Xf_t):.4f}  val rank {r:.4f}'
                  f'{"  *" if r == best else ""}')
    head.load_state_dict(best_state)
    return head, best


if RUN_CLIP_PROBE:
    t0 = time.time()
    print('Section A: ClipLoss on the frozen pooled features')
    headA, bestA = train_head(Xtr[~is_val], ytr[~is_val], Xtr[is_val], ytr[is_val],
                              A_EPOCHS, A_BATCH, A_LR)
    with torch.no_grad():
        predA = headA(torch.tensor(Xte, device=DEVICE)).cpu().numpy()
    print(f'  best val rank {bestA:.4f}   ({time.time() - t0:.0f}s)')
    resA = score(predA, yte, ste, emb, 'ClipLoss probe')
    print('  ridge baseline   rank 0.4998   top-5  2.48% (per-window)   2.50% (subject-agg)')
    torch.save({k: v.cpu() for k, v in headA.state_dict().items()}, WEIGHTS / 'head_clip.pt')
else:
    print('Section A skipped')

## Section B — LoRA on the attention projections

The encoder now trains, so the cached features are void and the windows are rebuilt from
the EDFs using the settings recorded in the bundle.

In [ ]:
import mne, pandas as pd
mne.set_log_level('ERROR')
STIM = {'train': 'stim_train', 'test': 'stim_test'}


def build_windows(rel, split):
    """Re-epoch one recording exactly as track1_02_features did."""
    sf = PP['sfreq']
    n_times = int(round(1.0 * sf))
    tmin, tmax = -0.2, -0.2 + 1.0 - 1.0 / sf
    raw = mne.io.read_raw_edf(RAW / DATASET / rel, preload=True)
    if PP['filt']:
        raw.filter(PP['filt'][0], PP['filt'][1], verbose=False)
    raw.resample(sf)
    if PP['scale_scope'] == 'recording':
        raw.apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-8), channel_wise=True)

    tsv = RAW / DATASET / rel.replace('_eeg.edf', '_events.tsv')
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM[split]].dropna(subset=['stim_file']).reset_index(drop=True)
    s = np.round(df.onset.values * sf).astype(int)
    lo, hi = int(round(tmin * sf)), int(round(tmax * sf))
    k = (s + lo >= 0) & (s + hi < raw.n_times)
    df, s = df[k].reset_index(drop=True), s[k]
    ev = np.column_stack([s, np.zeros_like(s), np.ones_like(s)])
    baseline = None if PP['baseline'] in ('None', 'none') else eval(PP['baseline'])
    epo = mne.Epochs(raw, ev, tmin=tmin, tmax=tmax, baseline=baseline,
                     preload=True, verbose=False)
    X = epo.get_data(copy=True).astype('float32')
    if PP['scale_scope'] == 'window':
        X = (X - X.mean(2, keepdims=True)) / (X.std(2, keepdims=True) + 1e-8)
    return np.clip(X, -PP['clamp'], PP['clamp']), df.stim_file.values[epo.selection]


if RUN_LORA:
    man = json.loads((PREP / 'prepare_manifest.json').read_text())
    key = {k: i for i, k in enumerate(t['keys'].tolist())}
    W = {'train': [[], [], []], 'test': [[], [], []]}
    t0 = time.time()
    for rel, meta_r in sorted(man['recordings'].items()):
        sp = meta_r['split']
        Xw, stim = build_windows(rel, sp)
        idx = np.array([key.get(s, -1) for s in stim])
        keep = idx >= 0
        W[sp][0].append(Xw[keep]); W[sp][1].append(idx[keep])
        W[sp][2].append(np.full(int(keep.sum()), meta_r['subject']))
    Wtr = [np.concatenate(a) for a in W['train']]
    Wte = [np.concatenate(a) for a in W['test']]
    CH = mne.io.read_raw_edf(RAW / DATASET / sorted(man['recordings'])[0],
                             preload=False).ch_names
    print(f'windows  train {Wtr[0].shape}  test {Wte[0].shape}   ({time.time() - t0:.0f}s)')
    print(f'memory   {(Wtr[0].nbytes + Wte[0].nbytes) / 1e9:.2f} GB float32')
else:
    print('Section B skipped')

In [ ]:
class LoRALinear(nn.Module):
    """Frozen base plus a rank-r update. B starts at zero, so training begins
    exactly at the pretrained function."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.zeros(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale


class RevePooledTrain(nn.Module):
    def __init__(self, chs_info, n_outputs, rank, alpha):
        super().__init__()
        from braindecode.models import REVE
        self.net = REVE.from_pretrained(ENCODER, n_outputs=n_outputs,
                                        n_chans=len(chs_info), n_times=200,
                                        sfreq=200.0, chs_info=chs_info)
        self.net.final_layer = nn.Identity()
        for p in self.net.parameters():
            p.requires_grad_(False)
        targets = [n for n, m in self.net.named_modules()
                   if isinstance(m, nn.Linear) and n.split('.')[-1] in ('to_qkv', 'to_out')]
        for name in targets:
            parent = self.net.get_submodule(name.rsplit('.', 1)[0])
            setattr(parent, name.split('.')[-1],
                    LoRALinear(self.net.get_submodule(name), rank, alpha))
        self.n_lora = len(targets)
        self.head = nn.Linear(self.net.embed_dim, n_outputs)

    def forward(self, X):
        feats = self.net(X, return_features=True)['features']
        return self.head(feats.mean(dim=(1, 2)))

In [ ]:
if RUN_LORA:
    model = RevePooledTrain([{'ch_name': c} for c in CH], DIM,
                            LORA_RANK, LORA_ALPHA).to(DEVICE)
    if RUN_CLIP_PROBE:
        model.head.load_state_dict(headA.state_dict())   # warm start, per the reference
        print('head warm-started from Section A')
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_tr = sum(p.numel() for p in trainable)
    n_all = sum(p.numel() for p in model.parameters())
    print(f'LoRA on {model.n_lora} projections   trainable {n_tr / 1e6:.2f}M '
          f'of {n_all / 1e6:.1f}M ({n_tr / n_all:.1%})')

    ytr_w, subj_w = Wtr[1], Wtr[2]
    iv = np.array([y in val_imgs for y in ytr_w])
    Xf = torch.tensor(Wtr[0][~iv]); yf = torch.tensor(ytr_w[~iv].astype('int64'))
    Xv = torch.tensor(Wtr[0][iv]);  yv = ytr_w[iv]
    print(f'fit {len(Xf):,} windows / val {len(Xv):,}')

    opt = torch.optim.AdamW(
        [{'params': [p for n, p in model.named_parameters()
                     if p.requires_grad and not n.startswith('head')], 'lr': B_LR},
         {'params': model.head.parameters(), 'lr': B_HEAD_LR}], weight_decay=1e-2)

    @torch.no_grad()
    def infer(X, bs=64):
        model.eval()
        return np.concatenate([model(X[i:i + bs].to(DEVICE)).float().cpu().numpy()
                               for i in range(0, len(X), bs)])

    best, best_state = -1, None
    for ep in range(1, B_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xf))
        tot, t0 = 0.0, time.time()
        for i in range(0, len(perm) - 1, B_BATCH):
            idx = perm[i:i + B_BATCH]
            if len(idx) < 8:
                continue
            opt.zero_grad()
            loss = clip_loss(model(Xf[idx].to(DEVICE)), yf[idx].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(idx)
        r = rank_pct(infer(Xv), yv, emb)
        if r > best:
            best = r
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        print(f'  epoch {ep:2d}  loss {tot / len(Xf):.4f}  val rank {r:.4f}'
              f'{"  *" if r == best else ""}  {time.time() - t0:.0f}s')
    model.load_state_dict(best_state)
    resB = score(infer(torch.tensor(Wte[0])), Wte[1], Wte[2], emb, 'LoRA')
    if RUN_CLIP_PROBE:
        print(f'  Section A was      rank {resA["rank"]:.4f}   top-5 {resA["top5"]:.2f}%')
    torch.save(best_state, WEIGHTS / 'lora_model.pt')

## Package the better of the two

In [ ]:
if MAKE_ZIP and RUN_LORA:
    best_res = resB if not RUN_CLIP_PROBE or resB['top5'] >= resA['top5'] else resA
    print(f'best per-window top-5: {best_res["top5"]:.2f}% vs chance {best_res["chance"]:.2f}%')
    if best_res['top5'] <= best_res['chance'] and not globals().get('FORCE_ZIP'):
        print('At or below chance; not building a zip. Set FORCE_ZIP = True to override.')
    else:
        print('Packaging is in track1_03_head for the frozen head. For LoRA the solver')
        print('must rebuild the LoRA modules before load_state_dict, so it needs its own')
        print('submission.py. Build that once these numbers justify a submission.')